# Implementing a simple AI Agent

In this lab, we will implement a simple AI agent that is able to retrive information from a datasource. This will be done by a mechanic called "Tool calling". 

Let us demystify the concept of an AI Agent. An AI agent is nothig more (**or less**) than a software program build around an LLM that allows the LLM to take autonomous decisions and use tools. Let us break this down:
- A "tool" is an object that allow LLMs to interact with external environments. These tools may be functions made available to LLMs such as a calculator, a search engine, or even another AI—to perform specific tasks it isn't designed to handle on its own. The tools can be executed separately whenever the LLM determines that their use is appropriate.
- A LLM trained to perform supports tool calling (more on this later). Basically, this means that the LLM is able to decide when to use a "tool" .
- Tool calling enables the model to generate a response for a prompt that aligns with a user-defined schema for a function.


that is able to interact with its environment. This interaction can be done in many ways, such as:

## Installation

### On Kaggle
Repeat the instructions from the previous part of the lab to install and run Ollama on Kaggle.
The last instruction should print: 

```python
print("Ollama server is running on 127.0.0.1:11438")
```

In [ ]:
# Pull a model (e.g., llama)
print("Pulling the llama model...")
subprocess.run(['ollama', 'pull', 'llama3.1:8b'])

### On your local machine
Be sure that Ollama is runnung (follow the instructions in the prrevious lab). In the terminal, pull the `llama3.1:8b` model or another [tool model](https://ollama.com/search?c=tools). Start with small models. The code below has been tested with `llama3.1:8b`  (**llama3.1:8b is about 5GB**). Other models may require some adjustments in the prompt. Check the documentation of the model you want to use or test other models.

```bash
ollama run llama3.1:8b
```

---

## Teach your model to do a simple calculation (example)
Original code from the [Ollama repository](https://github.com/ollama/ollama-python/blob/main/examples/tools.py). 

### Step 1: Import the necessary libraries

We use `ChatResponse` and `chat`.

In [1]:
from ollama import ChatResponse, chat

### Step 2: Define the functions (tools)

In this example, we define two functions: `add_two_numbers` and `subtract_two_numbers`. These functions will be used by the model to perform simple calculations.

Basically, each function defines a prompt for the model and what the model should do with the input. As you can see in the example, the prompte structure is does not need to be complex or always the same.

In [ ]:
# We strictly define the types of the arguments and the return value to be sure that the tool will work as expected
def add_two_numbers(a: int, b: int) -> int:
  """
  Add two numbers

  Args:
    a (int): The first number
    b (int): The second number

  Returns:
    int: The sum of the two numbers
  """

  # The cast is necessary as returned tool call arguments don't always conform exactly to schema
  # E.g. this would prevent "what is 30 + 12" to produce '3012' instead of 42
  return int(a) + int(b)


def subtract_two_numbers(a: int, b: int) -> int:
  """
  Subtract two numbers
  """

  # The cast is necessary as returned tool call arguments don't always conform exactly to schema
  return int(a) - int(b)




In the examples above, the tools are defined as functions. However, tools can also be manually defined as dictionaries (see below).

In [ ]:
# Tools can still be manually defined and passed into chat
subtract_two_numbers_tool = {
  'type': 'function',
  'function': {
    'name': 'subtract_two_numbers',
    'description': 'Subtract two numbers',
    'parameters': {
      'type': 'object',
      'required': ['a', 'b'],
      'properties': {
        'a': {'type': 'integer', 'description': 'The first number'},
        'b': {'type': 'integer', 'description': 'The second number'},
      },
    },
  },
}

In [4]:
# We create a dictionary that maps the function names to the functions

available_functions = {
  'add_two_numbers': add_two_numbers,
  'subtract_two_numbers': subtract_two_numbers,
}

### Step 3: Define the chat with the models.
We define the message, we inform the model about the available tools, and we call the model.

In [35]:
messages = [{'role': 'user', 'content': 'What is three plus one?'}]
print('Prompt:', messages[0]['content'])

response: ChatResponse = chat( # here the `:` means that response is of type ChatResponse
  'llama3.1:8b', # change this to the model you want to use
  messages=messages,
  tools=[add_two_numbers, subtract_two_numbers_tool], # we indicate the tools available to the model
)

Prompt: What is three plus one?


### Step 4: We process the response

The idea is that the model decides whether to use the tool or not. The choice of the model is stored in the variable `response.message.tool_calls`. ToolCall is a list of the tools that the models decided to use.

In [36]:
print('Response:', response.message.tool_calls) # print this for debugging/undersanding the tool calls

Response: [ToolCall(function=Function(name='add_two_numbers', arguments={'a': 3, 'b': 1}))]


**Question**: The response contains the tool(s) to call with its arguments. Try to ask a question that is not related to the tools. What does the model do? 

**Answer**: The response.message.tool_calls list will be empty or propose non-existent tools.

In the code below, we test whether the model decided to use one or several tools.

NOTES:
- The `:=` operator, also known as the [*walrus* operator](https://realpython.com/python-walrus-operator/), is used for assignment expressions. It allows you to assign a value to a variable as part of an expression.
Here below, we call `available_functions.get(tool.function.name)` and we assign the result to the variable `function_to_call` in one line.
- The `**` operator is used to pass a dictionary as keyword arguments.
In summary, in the code below, we assign to `function_to_call` the function that the model decided to use (using the `:=` operator) and we pass the arguments to the function (using the `**` operator).

In [37]:
# we check if the model returned any tool calls
if response.message.tool_calls:
  # There may be multiple tool calls in the response, we treat them one by one ina for loop
  for tool in response.message.tool_calls:
    # Ensure the function is available, and then call it
    function_to_call = available_functions.get(tool.function.name)
    if function_to_call:
      print('Calling function:', tool.function.name)
      print('Arguments:', tool.function.arguments)
      # The ** operator unpacks the dictionary into keyword arguments
      # If you did not change the example, the instruction below will be equivalent to add_two_numbers(a=3, b=1)
      output = function_to_call(**tool.function.arguments)
      print('Function output:', output)
    else:
      print('Function', tool.function.name, 'not found')



Calling function: add_two_numbers
Arguments: {'a': 3, 'b': 1}
Function output: 4


### Step 5: The final "ribbon"	
We use the LLM once again to process the response and print the result in a nice way.

In [38]:
# Only needed to chat with the model using the tool call results
if response.message.tool_calls:
  # Add the function response to messages for the model to use
  messages.append(response.message)
  messages.append({'role': 'tool', 'content': str(output), 'name': tool.function.name})

else: 
  print('No tool calls returned from model, just providing the response to the user...')
  # we do not need to modify the messages if the model did not return any tool calls

# Get final response from model with function outputs
final_response = chat('llama3.1:8b', messages=messages)
print('Final response:', final_response.message.content)


Final response: The answer to the equation 3 + 1 is 4.
